# Alcantara Studio — MuseTalk no Kaggle

Este notebook é o caminho de processamento para o notebook do Cláudio **sem exigir GPU NVIDIA local**.

O computador local só precisa de navegador. O processamento usa o GPU gratuito do Kaggle. A disponibilidade é limitada, mas o Kaggle informa atualmente acesso gratuito a GPU NVIDIA e quota semanal de GPU (em geral 30 h ou mais, conforme demanda). O notebook usa MuseTalk 1.5 e gera MP4.

**Importante:** o notebook não depende do Docker/NVIDIA do Windows.

In [ ]:
import os, sys, subprocess, shutil, json, time, textwrap
print('Python:', sys.version)
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader']).decode())

## 1. Preparar o ambiente

A instalação fica em `/kaggle/working`, portanto não altera o Windows. A versão Python 3.10 é usada porque foi a combinação validada no teste anterior.

In [ ]:
!pip -q install uv
!uv python install 3.10
!uv venv /kaggle/working/musetalk-env --python 3.10
PY='/kaggle/working/musetalk-env/bin/python'
PIP='/kaggle/working/musetalk-env/bin/pip'
print('Ambiente:', PY)

In [ ]:
import os
os.environ['PYTHON_BIN']='/kaggle/working/musetalk-env/bin/python'
PY=os.environ['PYTHON_BIN']
!$PY -m pip install -q --upgrade pip
!$PY -m pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118


In [ ]:
!rm -rf /kaggle/working/MuseTalk
!git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
%cd /kaggle/working/MuseTalk
!$PY -m pip install -q -r requirements.txt
!$PY -m pip install -q openmim==0.3.9
!$PY -m pip install -q mmengine
!$PY -m mim install mmcv==2.0.1
!$PY -m mim install mmdet==3.1.0
!$PY -m mim install mmpose==1.1.0
!$PY -m pip install -q gdown==4.7.3 huggingface_hub==0.30.2
print('Dependências instaladas.')

In [ ]:
%cd /kaggle/working/MuseTalk
!chmod +x download_weights.sh
!bash download_weights.sh
print('Modelos baixados/verificados.')

## 2. Envie seu vídeo e sua música

Clique nos dois botões abaixo. O vídeo precisa cobrir a duração da música. Para o primeiro teste, use um vídeo curto. Depois podemos usar a versão de 4:17.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
video_upload = widgets.FileUpload(accept='.mp4,.mov,.mkv', multiple=False, description='Vídeo')
audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.aac', multiple=False, description='Música')
display(video_upload)
display(audio_upload)


In [ ]:
from pathlib import Path
import subprocess, os
def save_upload(widget, target_dir):
    if not widget.value:
        raise RuntimeError('Arquivo não enviado. Volte à célula anterior e selecione o arquivo.')
    item = widget.value[0]
    name = Path(item['name']).name
    path = Path(target_dir) / name
    path.write_bytes(item['content'].tobytes())
    return str(path)

WORK=Path('/kaggle/working/alcantara-input')
WORK.mkdir(exist_ok=True)
VIDEO=save_upload(video_upload, WORK)
AUDIO=save_upload(audio_upload, WORK)
print('Vídeo:', VIDEO)
print('Áudio:', AUDIO)

In [ ]:
def duration(path):
    out=subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',path])
    return float(out.decode().strip())
vd=duration(VIDEO); ad=duration(AUDIO)
print(f'Vídeo: {vd:.2f}s')
print(f'Áudio: {ad:.2f}s')
if vd + 0.5 < ad:
    raise RuntimeError('O vídeo é mais curto que a música. Use um vídeo-base que cubra toda a faixa.')

## 3. Executar o MuseTalk 1.5

Esta célula usa a mesma lógica de inferência que foi validada no teste anterior. O resultado será salvo em `/kaggle/working/MuseTalk/results/v15/`.

In [ ]:
from pathlib import Path
import yaml, os, subprocess, time
mt=Path('/kaggle/working/MuseTalk')
cfg=mt/'configs/inference/alcantara_kaggle.yaml'
cfg.write_text(yaml.safe_dump({'task_0': {'video_path': VIDEO, 'audio_path': AUDIO, 'result_name': 'alcantara_studio.mp4'}}, sort_keys=False), encoding='utf-8')
cmd=[
    PY, 'scripts/inference.py',
    '--inference_config', str(cfg),
    '--unet_config', './models/musetalkV15/musetalk.json',
    '--unet_model_path', './models/musetalkV15/unet.pth',
    '--whisper_dir', './models/whisper',
    '--result_dir', './results',
    '--version', 'v15',
    '--use_float16',
    '--batch_size', '4'
]
env=os.environ.copy(); env['PYTHONPATH']=str(mt); env['MPLBACKEND']='Agg'
started=time.time()
subprocess.run(cmd, cwd=mt, env=env, check=True)
print(f'Tempo total: {time.time()-started:.1f}s')

In [ ]:
from IPython.display import FileLink, display
result=Path('/kaggle/working/MuseTalk/results/v15/alcantara_studio.mp4')
if not result.exists():
    candidates=list((mt/'results'/'v15').glob('*.mp4'))
    print('Resultados encontrados:', candidates)
else:
    print('RESULTADO PRONTO:', result)
    display(FileLink(str(result)))

## 4. Depois do teste

Se o vídeo curto sair corretamente, repita o notebook com o vídeo de 4:17 e a música de 4:17. O notebook não precisa do seu computador para a inferência; o navegador só envia os arquivos e baixa o MP4 final.

O Kaggle oferece GPU gratuita, mas a quota e a disponibilidade não são garantidas. Portanto, este é o **plano gratuito de processamento**, não uma promessa de GPU ilimitada.